# gemma-4-12b-it expansion eval (run 027)
Serves unsloth/gemma-4-12b-it-GGUF Q4_K_M tuned on CPU, runs the same 45-IR
expansion set with per-call timing capture, saves replies-gemma12-fast.jsonl.
Scoring happens LOCALLY. Settings: no accelerator, Internet ON.


In [ ]:
import subprocess
import sys


def pip_install(*args):
    print('+', 'pip install', ' '.join(args), flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                     check=True)


pip_install('--upgrade', 'pip')
pip_install('llama-cpp-python')
# Server runtime closure (bare install lacks it; v3/v2 died on sse_starlette).
pip_install('llama-cpp-python[server]')
pip_install('sse-starlette', 'starlette-context', 'fastapi',
            'uvicorn[standard]', 'pydantic', 'pydantic-settings',
            'anyio', 'httpx')
pip_install('huggingface_hub', 'hf_transfer')
import llama_cpp
import sse_starlette, starlette_context, fastapi, uvicorn  # noqa: F401
from llama_cpp.server.app import create_app  # noqa: F401
print('server closure OK (import chain verified pre-download)')
print('llama-cpp-python OK (CPU build)')


In [ ]:
import json as _json
import sys as _sys
import urllib.request as _urlreq

TUNNEL = ""  # stamped by phonehome/push.sh (public receiver URL); empty = disabled
TOKEN = ""  # stamped by phonehome/push.sh (bearer token); empty = disabled
KID = 'cpu-gemma12-fast'  # per-kernel short id (cpu-shootout, cpu-lfm25, cpu-gemma12, ...)


def hb(phase, msg=""):
    if not TUNNEL or not TOKEN:
        return
    try:
        body = _json.dumps(
            {"kernel": KID, "phase": phase, "msg": str(msg)[:300]}
        ).encode()
        req = _urlreq.Request(
            f"{TUNNEL}/hb?token={TOKEN}",
            data=body,
            headers={"Content-Type": "application/json"},
        )
        with _urlreq.urlopen(req, timeout=5):
            pass
    except Exception:
        pass


_prev_hook = _sys.excepthook


def _hook(t, v, tb, _p=_prev_hook):
    hb("fatal", f"{t.__name__}: {v}")
    return _p(t, v, tb)


_sys.excepthook = _hook
print("phone-home:", "ON" if TUNNEL and TOKEN else "OFF (unstamped)")

import json
import os
import subprocess
import sys
import time
import urllib.request

REPO = 'unsloth/gemma-4-12b-it-GGUF'
GGUF_FILE = 'gemma-4-12b-it-Q4_K_M.gguf'
N_CTX = 2048
PORT = 8085
TEMP = 0
MAX_TOKENS = 512

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download

D = json.load(open('/kaggle/input/datasets/blonded2/qwen-eval-irs/irs.json'))
P = json.load(open('/kaggle/input/datasets/blonded2/qwen-eval-irs/prompts.json'))
assert len(D) == 45, len(D)
PROMPT = P['v0']
assert '{draft}' in PROMPT and 'TESTDRAFT' not in PROMPT
print('eval set OK:', len(D), 'IRs, single arm')
hb('eval-ok', f'{len(D)} IRs')

hb('downloading', GGUF_FILE)
path = hf_hub_download(repo_id=REPO, filename=GGUF_FILE)
print('downloaded:', path, flush=True)
hb('downloaded', GGUF_FILE)

log = open('/tmp/llama-server.log', 'w')
help_txt = subprocess.run([sys.executable, '-m', 'llama_cpp.server', '--help'], capture_output=True, text=True).stdout
for flag in ['--n_threads', '--n_threads_batch', '--n_batch', '--n_ubatch', '--type_k', '--type_v', '--use_mlock', '--flash_attn']:
    assert f' {flag}' in help_txt or f'\n{flag}' in help_txt, f'{flag} missing from server --help'
EXTRA = ['--n_threads', '4', '--n_threads_batch', '4', '--n_batch', '512', '--n_ubatch', '512',
         '--type_k', '8', '--type_v', '8', '--use_mlock', 'true', '--flash_attn', 'true']
print('tuned flags:', ' '.join(EXTRA), flush=True)
cmd = [sys.executable, '-m', 'llama_cpp.server',
       '--model', path, '--n_gpu_layers', '0',
       '--n_ctx', str(N_CTX), '--host', '127.0.0.1',
       '--port', str(PORT)] + EXTRA
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
for i in range(60):
    if proc.poll() is not None:
        print('--- server died; log tail ---')
        print(''.join(open('/tmp/llama-server.log').readlines()[-40:]))
        print('--- env capsule ---')
        print(subprocess.run('free -g; df -h /tmp | tail -1', shell=True, capture_output=True, text=True).stdout)
        raise RuntimeError(f"server exited with code {proc.returncode}")
    time.sleep(10)
    try:
        with urllib.request.urlopen(
                f'http://127.0.0.1:{PORT}/v1/models', timeout=5) as r:
            print('READY, status', r.status, flush=True)
            break
            hb('ready', 'gemma12')
    except Exception:
        if i % 6 == 0:
            print(f'waiting for server... ({i * 10}s)', flush=True)
else:
    raise RuntimeError('server did not come up in 10 min')

outp = '/kaggle/working/replies-gemma12-fast.jsonl'
n_err = 0
toks = []
with open(outp, 'w') as out:
    for i, item in enumerate(D):
        prompt = PROMPT.replace('{draft}', item['ir'])
        body = json.dumps({
            'messages': [{'role': 'user', 'content': prompt}],
            'temperature': TEMP, 'max_tokens': MAX_TOKENS}).encode()
        row = {'key': item['key'], 'reply': '', 'finish': '',
               'wallMs': 0, 'tokPerSec': None, 'compTokens': None, 'error': None}
        ok = False
        for attempt in range(3):
            try:
                t0 = time.time()
                req = urllib.request.Request(
                    f'http://127.0.0.1:{PORT}/v1/chat/completions',
                    data=body, headers={'Content-Type':
                                        'application/json'})
                with urllib.request.urlopen(req, timeout=900) as r:
                    d = json.load(r)
                row['wallMs'] = round((time.time() - t0) * 1000)
                row['reply'] = d['choices'][0]['message']['content'] or ''
                row['finish'] = d['choices'][0]['finish_reason']
                tm = d.get('timings') or {}
                row['tokPerSec'] = tm.get('predicted_per_second')
                usg = d.get('usage') or {}
                row['compTokens'] = usg.get('completion_tokens')
                toks.append(row["tokPerSec"]) if row["tokPerSec"] else None
                ok = True
                break
            except Exception as e:
                row['error'] = f'{type(e).__name__}: {e}'[:200]
                time.sleep(10)
        if not ok:
            n_err += 1
        out.write(json.dumps(row) + '\n')
        out.flush()
        if (i + 1) % 10 == 0:
            print(f'{i + 1}/45, {n_err} errors', flush=True)
            hb('progress', f'{i + 1}/45 {n_err}err' + (f' {sum(toks)/len(toks):.1f}t/s' if toks else ''))
        assert n_err <= 10, f'error budget exceeded ({n_err})'
print('done')
hb('done', 'gemma12 complete')
